# Group 42

Main

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

from thymio import Thymio
from vision import Vision

import motion_controll
import filtering
import local_nav
import global_nav
import utils

In [ ]:
thymio = Thymio(pos_init=[0, 0], orient=0)
await thymio._connect_to_thymio_()
thymio.stop()

Local nav tests

In [ ]:
thymio.nav_mode = "LOCAL"
while True: 
    await thymio.update_ir()
    # is_object = local_nav.is_object(thymio)
    # if(is_object):
    #     print("object detected")
    print(str(thymio.ir_sensors) + "                     ", end="\r")
    # avoid_right = local_nav.avoid_right(thymio, grid)
    local_nav.avoid_obstacle(thymio, 0, 0, True)
# await thymio.update_ir()
# print(thymio.ir_sensors)

Motion controll testing

In [ ]:
thymio.pos = [20,20]
thymio.orient = 0
goal = [22,25]
print(motion_controll.follow_path(thymio, goal))

In [ ]:
await thymio.unlock()

## Main Loop

In [ ]:
thymio.stop()

In [ ]:
# External modules import
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

# Internal modules and thymio class import
from thymio import Thymio
from vision import Vision
import motion_controll
import filtering
import local_nav
import global_nav
import utils

# Constants
GL_NAV_CHANGE_THLD = 20  # Number of cycles without obstacle to switch back to global navigation
WAIT_AFTER_KIDNAP = 20

# Initalisation of the grid
visionInstance = Vision() # Calls getEnvironment which stores the arena and creates the grid
# visionInstance.display_grid()

# Get cell size for coordinate conversions
cell_size_cm = visionInstance.getCellSizeCm()

# Get start position and orientation
ret, frame = visionInstance.cap.read() #Taking a single image to find the start pos of the robot
if not ret:
    raise ValueError("Camera failed to capture the frame.")

#start_pos, start_orient = visionInstance.getRobotPose(frame)
start_pos, start_orient = visionInstance.getInitialRobotPose()
print(start_pos, start_orient)

# Convert to grid coordinates (start_pos is in meters, convert to cm first)
if start_pos:
    start_cell = utils.real_to_grid((start_pos[0]*100, start_pos[1]*100))
    # print(f"start: {start_cell}")
    goal_pos = visionInstance.getGoalPos()
    goal_cell = utils.real_to_grid((goal_pos[0]*100, goal_pos[1]*100))  # Goal in cm
    # print(f"goal: {goal_cell}")
    # visionInstance.display_grid(start_cell, goal_cell)

# Get the grid from vision
grid = visionInstance.grid

# Find path to goal (mode = "a_star" or "djikstra")
# 0 for djikstra, 1 for a*
path_find_mode = 1
path, expanded_grid = global_nav.find_path(path_find_mode, grid, start_cell, goal_cell)

global_nav.display_grid(expanded_grid, start_cell, goal_cell)

# Initialisation of the thymio and connexion
thymio = Thymio(pos_init=start_pos, orient=start_orient)
await thymio._connect_to_thymio_()
thymio.stop()

# loop variables initialisation
vis_cntr = 0

# Local nav variables
gl_nav_change_cntr = 0
kidnaped_cntr = 0
avoid_right = True
LN_orient = 0
LN_pos_at_obst = [0, 0]
LN_stage = 0
obstacle_avoided = False
is_kidnaped = False

# Trajectory tracking
robot_trajectory = []  # List to store (x_cm, y_cm) positions

# filter initialisation
x_est, P_est, Q, R = filtering.init_filter(utils.q_x, utils.q_y, utils.q_theta, utils.q_v, utils.r_x,
                                           utils.r_y, start_pos*100, start_orient) #carefull maybe start_pos in meters

# Initialisation of robot position data and global path for visualisations
cam_robot_positions_cm = []
filter_robot_positions_cm = []
global_path_points_cm = path.copy()

while(True):
    # print(f"Nav_mode: {thymio.nav_mode}")
    start_loop_time = time.time()
    await thymio.update_ir()
    await thymio.update_ground_sensors()

    # print(thymio.ir_sensors)
    is_object = local_nav.is_object(thymio)
    is_kidnaped = local_nav.check_kidnap(thymio)

    if(is_kidnaped):
        thymio.nav_mode = "KIDNAPPED"
        thymio.stop()

    # is_object = False
    if(is_object and thymio.nav_mode=="GLOBAL"):
        avoid_right = local_nav.avoid_right(thymio, grid)
        thymio.nav_mode = "LOCAL"
    
    if(obstacle_avoided):
        if(thymio.nav_mode=="GLOBAL"):
            print("Error: obstacle_avoided true in global nav")
            break
        else:
            obstacle_avoided = False
            LN_stage = 0
            thymio.nav_mode = "GLOBAL"
            robot_cell = utils.real_to_grid((thymio.pos[0], thymio.pos[1]))
            path, expanded_grid = global_nav.find_path(path_find_mode, grid, robot_cell, goal_cell)
            global_path_points_cm = path.copy()
    
    if(thymio.nav_mode == "KIDNAPPED" and not is_kidnaped):
        kidnaped_cntr += 1
        if kidnaped_cntr < WAIT_AFTER_KIDNAP:
            ret, frame = visionInstance.cap.read() 
            if not ret:
                print("Camera failed to capture the frame.")
                break

            vis = frame.copy()
            result = visionInstance.getRobotPoseAndVisualise(frame, vis)

            # Check if detection failed (returns (None, None))
            if result[0] is None:
                print("Robot not detected during kidnap wait")
            else:
                # print("Robot detected after kidnap wait")  
                # print("Results[0] from vision: ", result[0]) 
                kidnaped_cntr = 0
                thymio.nav_mode = "GLOBAL"
                thymio.pos[0] = result[0][0]*100
                thymio.pos[1] = result[0][1]*100
                thymio.orient = result[1]
                # print("Robot pos after kidnap: ", thymio.pos, " orient: ", thymio.orient)
                robot_cell = utils.real_to_grid((thymio.pos[0], thymio.pos[1]))
                path, expanded_grid = global_nav.find_path(path_find_mode, grid, robot_cell, goal_cell)
                global_path_points_cm = path.copy()
                print("Robot re-localized, switching to GLOBAL navigation")

    match thymio.nav_mode:
        case "KIDNAPPED":
            # Stop the robot
            thymio.stop()
        
        case "LOCAL":
            # local navigation to avoid obstacle
            obstacle_avoided, LN_stage, LN_pos_at_obst, LN_orient = local_nav.avoid_obstacle(thymio, avoid_right, LN_stage, LN_pos_at_obst, LN_orient)

        case "GLOBAL":
            next_wp = path[0]
            # next_wp = [35.0, 35.0]  # Copy to avoid modifying the original
            # print(f"thymio pos: {thymio.pos}, next wp: {next_wp}", end="\r")
            wp_reached = motion_controll.follow_path(thymio, next_wp)
            if(wp_reached):
                # print("Waypoint reached!")
                path.pop(0)  # Supprime le waypoint atteint
                if len(path) == 0:  # Si plus de waypoints
                    print("Goal reached")
                    thymio.stop()
                    break  # Sortir de la boucle

    
    # stores the last orientation
    thymio.last_orient = thymio.orient
    
    # pos_on_img, orient_on_img = vision.get_pos()
    # Get robot position from vision 
    ret, frame = visionInstance.cap.read() 
    if not ret:
        print("Camera failed to capture the frame.")
        break

    vis = frame.copy() # Copying frame so we can display shapes on top without affecting detection
    visionInstance.visualiseArena(vis, visionInstance.arena_corners_pixels)
    # visualise global nav path
    visionInstance.visualiseGlobalPath(vis, global_path_points_cm)

    # --- Always detect and draw robot pose ---
    result = visionInstance.getRobotPoseAndVisualise(frame, vis)
    
    # Check if detection failed (returns (None, None))
    if result[0] is None:
        x_est, P_est = filtering.filter_pos(thymio, [None, None, None, 0], x_est, # *100 to pu it in cm
                                        P_est, Q, R, 1/utils.FREQ_MAIN_LOOP, utils.RATIO_SPEED)
        # print("Robot not detected only filtering")
    else:
        [X_robot, Y_robot], robot_heading_angle = result
        # print(f"X: {X_robot:.5f}, Y: {Y_robot:.5f}, Direction: {robot_heading_angle:.5f}", end="\r")
        x_est, P_est = filtering.filter_pos(thymio, [X_robot*100, Y_robot*100, robot_heading_angle, 0], x_est, # *100 to pu it in cm
                                        P_est, Q, R, 1/utils.FREQ_MAIN_LOOP, utils.RATIO_SPEED)
        # print(f"Robot pos from cam :{X_robot*100:.2f} cm, {Y_robot*100:.2f} cm, orient: {robot_heading_angle:.2f} rad")
    
    
    # print(f"Robot pos from filter :{x_est[0]:.2f} cm, {x_est[1]:.2f} cm, orient: {x_est[2]:.2f} rad")


    thymio.pos = [x_est[0], x_est[1]]  # already in cm
    thymio.orient = x_est[2]

    # Visualise before and after filtering - only add every 5 points
    if (vis_cntr == 0 or vis_cntr % 10 == 0):
        if (result[0] is not None):
            cam_robot_positions_cm.append([result[0][0]*100, result[0][1]*100]) # Convert m to cm
        filter_robot_positions_cm.append([x_est[0], x_est[1]])
    visionInstance.visualiseGlobalPath(vis, filter_robot_positions_cm, (0, 215, 255))
    visionInstance.visualiseGlobalPath(vis, cam_robot_positions_cm, (255, 0, 0))
    vis_cntr += 1

    # Resize image for larger display (2x scale)
    vis_large = cv2.resize(vis, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LINEAR)
    cv2.imshow("Live camera", vis_large)
    if(cv2.waitKey(1) & 0xFF == ord('q')):
        thymio.stop()
        break

    # if(vis_cntr >= 5):
    #     vis_cntr = 0
    #     vis_large = cv2.resize(vis, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LINEAR)
    #     cv2.imshow("Live camera", vis_large)
    #     if(cv2.waitKey(1) & 0xFF == ord('q')):
    #         thymio.stop()
    #         break


    # filtering.filter_pos(thymio, pos_on_img, orient_on_img)
    time_main_loop = time.time() - start_loop_time
    print(f"Main loop time: {time_main_loop:.3f} s")
    if(time_main_loop < 1/utils.FREQ_MAIN_LOOP):
        # print(f"Wait for : {1/utils.FREQ_MAIN_LOOP - time_main_loop}")
        await asyncio.sleep(1/utils.FREQ_MAIN_LOOP - time_main_loop)
    # print(f"main loop time : {time_main_loop}")
    herz_main_loop = 1 / (time.time() - start_loop_time)
    print(f"Herz main loop: {herz_main_loop:.4f}")


while(True):
    if(cv2.waitKey(1) & 0xFF == ord('q')):
        cv2.destroyAllWindows()
        break


In [ ]:
thymio.stop()

In [ ]:

# Get robot pose in loop
if not visionInstance.cap.isOpened(): # Checking access to camera feed
    print("Error: Could not access the webcam.")
    exit()

while True:
    ret, frame = visionInstance.cap.read() # Taking an image frame from the camera feed
    if not ret:
        print("Camera failed to capture the frame.")
        break

    vis = frame.copy() # Copying frame so we can display shapes on top without affecting detection

    # --- Always redraw arena outline ---
    visionInstance.visualiseArena(vis, visionInstance.arena_corners_pixels)

    # --- Always detect and draw robot pose ---
    [X, Y], robot_heading_angle = visionInstance.getRobotPoseAndVisualise(frame, vis)
    if (X is None or Y is None or robot_heading_angle is None):
        continue
    print(f"X: {X:.5f}, Y: {Y:.5f}, Direction: {robot_heading_angle:.5f}")

    # --- Show the live window ---
    cv2.imshow("Live Robot Pose", vis)

    # Exit on Q
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

visionInstance.cap.release()
cv2.destroyAllWindows()

In [ ]:
a = [[1,2],[3,4]]
a[1][1]

In [ ]:
await thymio.update_ir()
print(thymio.ir_sensors[0:5])

In [ ]:
thymio.set_motor_speeds([100, 100])
while True:
    await thymio.update_ir()
    if sum(thymio.ir_sensors) > 2000:
        thymio.stop()
        break
    

Next cell makes the Thymio robot move forward for 4 seconds and then stops each time the Forward button is pressed. Program stops when the Backward button is pressed.  
It is intended to collect data for computing the **velocity variance**.

In [ ]:
await thymio.button_loop()

Using this program to measure (with a ruler) the distance travelled by the bot each time to see differencies despite constant time and compute the variance on speed state.

In [ ]:
import numpy as np
data_velocity_error=[142, 138, 138, 137, 139, 137, 138, 135, 142, 141] #distances in mm travelled at presumed same speed for a constant time
data_velocity_error=[x/4 for x in data_velocity_error] #distances divided by the constant time to get true velocities

mean_speed=np.mean(data_velocity_error) #mean speed in mm/s
print(mean_speed)
ratio_speed=100/mean_speed #from tests above, for a speed of 100 we get a mean speed of 34.675 mm/s

q_v = np.var(data_velocity_error) # variance on speed state
print(q_v)

<img src="position_measurement.png" width="400">

From the camera we got a data set of XY position measurements from the same position to search for some differencies and compute **variances on XY states and measurements**.

In [ ]:
measurements_from_camera=np.array([[13.118, 13.303], [13.153, 13.317], [13.090, 13.307], [13.062, 13.255], [13.059, 13.281],
                                   [13.074, 13.321], [13.026, 13.273], [13.073, 13.295], [13.023, 13.270]])
x_measurements = [x[0] for x in measurements_from_camera]
y_measurements = [y[1] for y in measurements_from_camera]

var_x = np.var(x_measurements)
var_y = np.var(y_measurements)

q_x = var_x/2 #variance on x position state
r_x = var_x/2 #variance on x position measurement
q_y = var_y/2 #variance on y position state
r_y = var_y/2 #variance on y position measurement
print(q_x, q_y, r_x, r_y)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

from thymio import Thymio
from vision import Vision

import motion_controll
import filtering
import local_nav
import global_nav
import utils

thymio = Thymio(pos_init=[0, 0], orient=0)
await thymio._connect_to_thymio_()
thymio.stop()

visionInstance = Vision()
visionInstance.display_grid()

ret, frame = visionInstance.cap.read()
print(ret)
start_theta= visionInstance.getInitialRobotPose()[1]
print(start_theta)
if start_theta is None:
    raise RuntimeError("Robot not detected in initial frame")

thymio.set_motor_speeds([0,0])
delta_t = 2
k=10
data_theta_pos=np.zeros(k)
data_theta_neg=np.zeros(k)

for i in range(k):

    thymio.set_motor_speeds([-100,100])
    start_time=time.time()
    while time.time() - start_time < delta_t:
        await asyncio.sleep(0.1)
    thymio.set_motor_speeds([0, 0])
    ret, frame = visionInstance.cap.read()
    vis=frame.copy()
    on_going_theta=visionInstance.getRobotPoseAndVisualise(frame, vis)[1]
    if on_going_theta is None:
        print("Robot not detected (positive turn), skipping this measurement")
        continue
    data_theta_neg[i]=on_going_theta-start_theta

    start_theta=on_going_theta
    thymio.set_motor_speeds([100,-100])
    start_time=time.time()
    while time.time() - start_time < delta_t:
        await asyncio.sleep(0.1)
    thymio.set_motor_speeds([0, 0])
    ret, frame = visionInstance.cap.read()
    vis=frame.copy()
    on_going_theta=visionInstance.getRobotPoseAndVisualise(frame, vis)[1]
    if on_going_theta is None:
        print("Robot not detected (positive turn), skipping this measurement")
        continue
    data_theta_pos[i]=on_going_theta-start_theta


print(f"theta neg : {data_theta_neg}")
print(f"theta pos : {data_theta_pos}")

In [ ]:
import numpy as np
data_theta=np.array([[1.20987582], [1.14751077], [1.14971828], [1.13941801], [1.12451291], [1.12512088],
                    [1.14892936], [1.13340664], [1.14468753], [1.14857364]])
q_theta=np.var(data_theta)
print(q_theta)

---
---

# **Group 42 Report**

---
---

# **Set up**


**Key Libraries:**  
- `numpy` - Array operations for the grid  
- `matplotlib` - Visualization of the map and path  
- `heapq` - Priority queue for efficient pathfinding  
- `cv2` (OpenCV) - Morphological operations to expand obstacles  
- `utils` - Project-specific constants and coordinate conversion functions

# **Environment**

Our environment is set up in the following way:
- white background
- aruco markers for robot detection, arena border detection and for the goal
- obstacles are cut out of red paper

We initially wanted to go with many different colours for the various detections but after testing, we realized that colours were quite difficult to detect and depended heavily on lighting, aruco markers were much more reliable.

---
---

# **Vision**

---
---

# **Global Navigation**

This part explains the global navigation module (`global_nav.py`) used for path planning in the Mobile Robotics Project.

The global navigation system finds a collision-free path from a **start** position to a **goal** position on a 2D occupancy grid. It uses:

1. **Obstacle Expansion** - Grow obstacles by the robot's size so we can treat the robot as a point  
2. **A* or Dijkstra Search** - Find the shortest path on the expanded grid  
3. **Path Simplification** - Reduce the cell-by-cell path to key waypoints  
4. **Visualization** - Display the results on the map

---

### 1. Occupancy Grid

The arena is represented as a **200x200 cell grid** (defined in `utils.py`):

| Value | Meaning |
|-------|---------|
| `0`   | Free space |
| `-1`  | Obstacle |

#### Coordinate Systems

There are two coordinate systems:

1. **Grid coordinates** `(row, col)`:
   - Row 0 = TOP, row 199 = BOTTOM  
   - Column 0 = LEFT, column 199 = RIGHT

2. **Real-world coordinates** `(x_cm, y_cm)`:
   - Origin (0, 0) = BOTTOM-LEFT  
   - X increases to the right, Y increases upward

Conversion functions in `utils.py`:
- `real_to_grid(coord)`
- `grid_to_real(coord)`
- `cm_to_cell(cm_value)`
- `cell_to_cm(cell_index)`

---

### 2. Obstacle Expansion

This function grows obstacles outward by the robot's size so that the pathfinding algorithm can treat the robot as a single point instead of checking its full footprint at every step.

The kernel size determines how much we expand the obstacle in all directions. We expand by half of the robot's size and add a small safety margin to make sure no overlap will occur between the obstacle and the robot.

---

### 3. Pathfinding Algorithm

The user can choose between 2 pathfinding algorithms. Both work using the 8 connected neighbouring cells on the grid.

#### Dijkstra

This algorithm assigns distances to each cell with start=0. Then, at each step, the unvisited node with the smallest known distance is picked. It has a greedy approach and explores cells outwards from the start until the goal is reached. The path can then be reconstructed by backtracking.

#### A star

**Heuristic Function**

- A* uses a heuristic to estimate distance to the goal. Because movement is 8-directional, we use the **octile distance**. 
(Reference for heuristic function choice : https://theory.stanford.edu/~amitp/GameProgramming/Heuristics.html)

- Diagonal moves cost `sqrt(2) = ~1.414`, straight moves cost `1`.  Optimal path: move diagonally as much as possible, then straight.

**Pathfinding**

A star works in the same way as Dijkstra's Algorithm but adds the heuristic function to guide the path finding towards the goal and therefore explore less cells.

==> We tested with both pathfinding algorithms and found that both work very well in our environment. The only noticeable difference was that A* gave us a smoother path.

---

### 4. Path Simplification

The path returned by the pathfinding algorithms give each cell the robot passes on. This is not efficient at all so we simplify this path by only taking the **corner waypoints**. 

The code looks at each cell in the path and checks the direction, if the direction of point 1 is the same as point 2, we remove point 2 from the list of cells. Then so on until we reach a point with a different direction, this one will be kept it in the simplified path list.


---
---

# **Local Navigation**

This part explains the local navigation module (`local_nav.py`) for real-time obstacle detection and avoidance.

The local navigation system handles **unexpected physical obstacles** not present in the map using:

1. **IR Sensor Detection** - Real-time obstacle detection
2. **Directional Decision** - Choose optimal avoidance side  
3. **Wall-Following** - Multi-stage avoidance maneuver  
4. **Kidnap Detection** - Detect robot lift-off

---

### 1. Navigation Modes

| Mode | Description |
|------|-------------|
| `GLOBAL` | Follows planned path |
| `LOCAL` | Reactive obstacle avoidance |
| `KIDNAPPED` | Robot lifted, stops and waits for re-localization |

---

### 2. Obstacle Detection

**IR Thresholds:**
```python
GLOBAL_IR_THLD = 4000   # Higher threshold for early detection
LOCAL_IR_THLD = 2000    # Lower threshold during avoidance
```

**Detection Function:**
```python
def is_object(thym: Thymio):
    ir_max = max(thym.ir_sensors)
    if(thym.nav_mode == "GLOBAL" and ir_max > GLOBAL_IR_THLD or
       thym.nav_mode == "LOCAL" and ir_max > LOCAL_IR_THLD):
        return True
    return False
```

---

### 3. Avoidance Direction

The `avoid_right()` function scans a rectangular region (30 cells forward × 50 cells lateral) and counts obstacles on each side.

**Direction Calculation:**
```python
# Forward direction
fwd_col = int(round(math.cos(orient)))
fwd_row = -int(round(math.sin(orient)))

# Right/left vectors
right_col = int(round(math.sin(orient)))
right_row = int(round(math.cos(orient)))
left_col, left_row = -right_col, -right_row
```

**Grid Scanning:**
```python
for fwd_dist in range(0, forward_depth + 1):
    for lateral_dist in range(1, lateral_distance + 1):
        # Count obstacles on right side
        if grid[right_row_check, right_col_check] == -1:
            obstacles_right += 1
        # Count obstacles on left side
        if grid[left_row_check, left_col_check] == -1:
            obstacles_left += 1
```

Returns `True` to avoid right (more obstacles left), `False` to avoid left.

---

### 4. Wall-Following Maneuver

**Parameters:**
```python
K_AVOID = 100           # Proportional gain
FWD_SPEED = 150         # Forward speed
ROT_SPEED = 90          # Rotation speed
ADVENCE_DIST = ROBOT_H  # Clearance distance
```

**5-Stage State Machine:**

**Stage 0 - Initial Rotation:** Rotate until obstacle is on chosen side
```python
if(ir_sens[0]==0 or sum(ir_sens[1:5])>0):
    thym.set_motor_speeds([ROT_SPEED, -ROT_SPEED])
else:
    stage=1
```

**Stage 1 - Wall Following:** Proportional control to maintain distance from obstacle
```python
if(ir_sens[0] > max(ir_sens[1:5])):
    thym.set_motor_speeds([FWD_SPEED, 
        int(FWD_SPEED-K_AVOID*max(ir_sens[1:5])/MAX_IR_VAL)])
else:
    thym.set_motor_speeds([int(FWD_SPEED-K_AVOID*ir_sens[0]/MAX_IR_VAL), 
        FWD_SPEED])
```

**Stage 2 - Clear Obstacle:** Move forward until distance = `4*ADVENCE_DIST/3`
```python
dis_from_obst = math.sqrt((thym.pos[0]-pos_at_obst[0])**2 + 
                          (thym.pos[1]-pos_at_obst[1])**2)
if(dis_from_obst >= 4*ADVENCE_DIST/3):
    stage=3
```

**Stage 3 - Rotate Back:** Turn to realign with path
```python
thym.set_motor_speeds([-ROT_SPEED, ROT_SPEED])
if(ir_sens[0]>0):
    stage=4
```

**Stage 4 - Final Advance:** Move forward `2*ADVENCE_DIST/3`, then return to GLOBAL
```python
if(dis_from_obst >= 2*ADVENCE_DIST/3):
    thym.stop()
    return True, 0, [0,0], 0
```

Symmetric implementation for both left/right avoidance (sensor 0 for right, sensor 4 for left).

---

### 5. Kidnap Detection

```python
KIDNAP_THRESHOLD = 100

def check_kidnap(thym: Thymio):
    if(max(thym.ground_sensors) < KIDNAP_THRESHOLD):
        return True
    return False
```

**Recovery:** Stop → Wait for placement → Vision re-localization → Replan path → Resume GLOBAL mode

---

### 6. Integration

```python
# Detect obstacle
is_object = local_nav.is_object(thymio)

# Switch to LOCAL
if(is_object and thymio.nav_mode == "GLOBAL"):
    avoid_right = local_nav.avoid_right(thymio, grid)
    thymio.nav_mode = "LOCAL"

# Execute avoidance
if(thymio.nav_mode == "LOCAL"):
    obstacle_avoided, LN_stage, LN_pos_at_obst, LN_orient = 
        local_nav.avoid_obstacle(thymio, avoid_right, LN_stage, 
                                LN_pos_at_obst, LN_orient)

# Return to GLOBAL
if(obstacle_avoided):
    thymio.nav_mode = "GLOBAL"
    robot_cell = utils.real_to_grid((thymio.pos[0], thymio.pos[1]))
    path = global_nav.find_path(path_find_mode, grid, robot_cell, goal_cell)
```

**Result:** Hybrid system combining global planning with local reactivity.

---
---

# **Filtering**

---
---

# **Motion Control**